# M4 — PyTorch Autograd Trace Lab

**Lab style:** Fill in the important details yourself. Observe what happens at each step.

## Learning Objectives

By the end of this lab, you will:
1. **Trace** the computational graph for a 2-layer MLP
2. **Inspect** how PyTorch stores gradients after `.backward()`
3. **Verify** that manual gradients match autograd
4. **Document** tensor shapes and gradient flow

## Prerequisites

- M1–M3 (gradients, backprop, optimizers)
- Basic PyTorch: `torch`, `nn`, tensor operations

---

## Lab Instructions

- **Sections with `# TODO` or `# YOUR CODE HERE`:** Complete these yourself.
- **Run cells in order** — the forward pass builds the graph; backward uses it.
- **Observe and document** — the reflection and table are as important as the code.

---
## Section 0: Setup

Import PyTorch and set a seed for reproducibility. Run this cell first.

In [1]:
import os
import torch
import torch.nn as nn

torch.manual_seed(42)

VIZ_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'visualizations'))
os.makedirs(VIZ_DIR, exist_ok=True)

---
## Section 1: Build a Minimal 2-Layer MLP

We use a simple architecture:

$$
z_1 = X W_1 + b_1, \quad a_1 = \sigma(z_1), \quad z_2 = a_1 W_2 + b_2, \quad \hat{y} = \sigma(z_2)
$$

Loss: BCE (binary cross-entropy).

**Your task:** Run the cell. Then in the next cell, **fill in** code to print the shape of each intermediate tensor (`z1`, `a1`, `z2`, `y_hat`) and whether it `requires_grad`. What do you observe?

In [2]:
# Fixed dimensions: 2 in, 3 hidden, 1 out
input_dim, hidden_dim, output_dim = 2, 3, 1
batch_size = 4

# Input and target (fixed for reproducible trace)
X = torch.tensor([[1.0, 2.0], [0.5, 1.0], [2.0, 0.5], [1.5, 1.5]], requires_grad=False)
y = torch.tensor([[1.0], [0.0], [1.0], [0.0]])

# Weights and biases (parameters we will take gradients of)
W1 = torch.randn(input_dim, hidden_dim, requires_grad=True)
b1 = torch.zeros(hidden_dim, requires_grad=True)
W2 = torch.randn(hidden_dim, output_dim, requires_grad=True)
b2 = torch.zeros(output_dim, requires_grad=True)

# Forward pass (manual, so we can inspect each step)
z1 = X @ W1 + b1
a1 = torch.sigmoid(z1)
z2 = a1 @ W2 + b2
y_hat = torch.sigmoid(z2)

loss_fn = nn.BCELoss()
loss = loss_fn(y_hat, y)

print(f"Loss: {loss.item():.4f}")

Loss: 0.9541


In [3]:
print("W1:", W1.shape, "requires_grad:", W1.requires_grad)
print("b1:", b1.shape, "requires_grad:", b1.requires_grad)
print("z1:", z1.shape, "requires_grad:", z1.requires_grad)
print("a1:", a1.shape, "requires_grad:", a1.requires_grad)
print("W2:", W2.shape, "requires_grad:", W2.requires_grad)
print("b2:", b2.shape, "requires_grad:", b2.requires_grad)
print("z2:", z2.shape, "requires_grad:", z2.requires_grad)
print("X:", X.shape, "requires_grad:", X.requires_grad)
print("y:", y.shape, "requires_grad:", y.requires_grad)
print("y_hat:", y_hat.shape, "requires_grad:", y_hat.requires_grad)
print("loss:", loss.shape, "requires_grad:", loss.requires_grad)

W1: torch.Size([2, 3]) requires_grad: True
b1: torch.Size([3]) requires_grad: True
z1: torch.Size([4, 3]) requires_grad: True
a1: torch.Size([4, 3]) requires_grad: True
W2: torch.Size([3, 1]) requires_grad: True
b2: torch.Size([1]) requires_grad: True
z2: torch.Size([4, 1]) requires_grad: True
X: torch.Size([4, 2]) requires_grad: False
y: torch.Size([4, 1]) requires_grad: False
y_hat: torch.Size([4, 1]) requires_grad: True
loss: torch.Size([]) requires_grad: True


---
## Section 2: Backward Pass and Gradient Inspection

**Save the graph first** (before backward frees it), then call `loss.backward()`. PyTorch traverses the computational graph **in reverse** and populates `.grad` on leaf tensors.

**Your task:** Run the save cell below, then the backward cell. After backward, print the gradients of W1, b1, W2, b2.

In [4]:
# Run backward (gradients flow from loss to parameters)
loss.backward()

In [5]:
print("W1.grad:", W1.grad)
print("b1.grad:", b1.grad)
print("W2.grad:", W2.grad)
print("b2.grad:", b2.grad)

W1.grad: tensor([[ 0.1379, -0.0260,  0.0333],
        [ 0.2040, -0.0462,  0.0478]])
b1.grad: tensor([ 0.1634, -0.0351,  0.0377])
W2.grad: tensor([[0.2088],
        [0.0668],
        [0.1604]])
b2.grad: tensor([0.3256])


---
## Section 3: Document the Computational Graph

**Your task:** Draw or describe the computational graph. Include:

1. **Nodes:** Input X, parameters (W1, b1, W2, b2), intermediates (z1, a1, z2, y_hat), loss
2. **Edges:** Operations (@, +, sigmoid)
3. **Direction:** Forward (data) and backward (gradients)

Use the markdown cell below to write your trace. You can use text, ASCII art, or note that you drew it on paper and saved to `visualizations/`.

In [10]:
# Save computational graph (run BEFORE loss.backward() — graph is freed after backward)
# Requires: pip install torchviz, and graphviz (e.g. brew install graphviz)
try:
    from torchviz import make_dot
    dot = make_dot(loss, params={'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2})
    dot.render(os.path.join(VIZ_DIR, 'm4_computational_graph'), format='png', cleanup=True)
    print(f"Saved to {VIZ_DIR}/m4_computational_graph.png")
except ImportError:
    print("Install torchviz: pip install torchviz. Also need graphviz: brew install graphviz")

Saved to /Users/bhavik.sanghvi/Desktop/llm-engineering-mastery/visualizations/m4_computational_graph.png


---
## Section 4: Tensor and Gradient Shape Table

**Your task:** Fill in the table below with the shapes you observed. (Re-run Section 1 and 2 if needed.)

| Tensor | Shape | Gradient (.grad) Shape |
|--------|-------|------------------------|
| X      | (4, 2) | None (not leaf with grad) |
| z1     | (4, 3) | — (intermediate) |
| a1     | (4, 3) | — |
| z2     | (4, 1) | — |
| y_hat  | (4, 1) | — |
| W1     | (2, 3) | (2, 3) |
| b1     | (3,) | (3, ) |
| W2     | (3, 1) | (3, 1) |
| b2     | (1,) | (1, ) |
| loss   | () | 1 (dloss/dloss) |

---
## Section 5: Manual Gradient Verification

**Your task:** Compute ∂L/∂W2 by hand (or with a small NumPy script) for the same forward pass. Compare with `W2.grad`.

**Hint:** For BCE + sigmoid, the gradient at the output is $\frac{\partial L}{\partial z_2} = \hat{y} - y$. Then $\frac{\partial L}{\partial W_2} = a_1^T \frac{\partial L}{\partial z_2}$ (averaged over batch).

Use the cell below to compute the manual gradient and compare.

In [7]:
with torch.no_grad():
    dL_dz2 = y_hat - y  # for BCE+sigmoid; shape (batch, 1)
    batch_size = X.shape[0]
    dL_dW2_manual = (a1.T @ dL_dz2) / batch_size  # (3,1) = W2 shape
    print("Manual dL/dW2:", dL_dW2_manual)
    print("Autograd W2.grad:", W2.grad)
    print("Match:", torch.allclose(dL_dW2_manual, W2.grad))

Manual dL/dW2: tensor([[0.2088],
        [0.0668],
        [0.1604]])
Autograd W2.grad: tensor([[0.2088],
        [0.0668],
        [0.1604]])
Match: True


---
## Section 6: What Happens After .backward()?

**Your task:** Run `.backward()` a second time without re-running the forward pass. What happens? Why?

Then try `loss.backward(retain_graph=True)` on a fresh run, and call `.backward()` again. What changes?

In [8]:
# When you call loss.backward() a second time without recreating the graph,
# PyTorch throws an error: tensors needed for gradient calculation were already freed.
# This is because by default, after .backward(), saved intermediates are released to save memory.

try:
    loss.backward()  # second backward call, triggers error
except RuntimeError as e:
    print("Expected RuntimeError:", e)
    print("\nThis happens because tensors needed for gradients are freed after the first backward pass.")
    print("To allow multiple backward passes, use backward(retain_graph=True) on the first call.")

Expected RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

This happens because tensors needed for gradients are freed after the first backward pass.
To allow multiple backward passes, use backward(retain_graph=True) on the first call.


---
## Section 7: Reflection

**Answer in your journal or below:**

1. When does PyTorch free the computational graph?
2. Why do only leaf tensors with `requires_grad=True` get `.grad` populated?
3. How would you implement a custom operation (e.g., a layer PyTorch doesn't have) so that gradients flow through it?

---

## Lab Completion Checklist

Before considering this lab done, ensure you have:

- [x] Printed shapes and `requires_grad` for z1, a1, z2, y_hat
- [x] Printed `.grad` for W1, b1, W2, b2 after `backward()`
- [x] Filled in the computational graph trace (Section 3)
- [x] Filled in the tensor/gradient shape table (Section 4)
- [x] Verified manual ∂L/∂W2 matches `W2.grad`
- [x] Experimented with second `backward()` call (and optionally `retain_graph=True`)
- [x] Answered the reflection questions (here or in journal)